In [ ]:
# ---------------------------------------------------------------------
# CELL 1: PASTE ALL OF THIS CODE INTO THE FIRST COLAB CELL AND RUN IT
# ---------------------------------------------------------------------

#@title A-to-Z Training Script for Neural Codec (v6)
#@markdown ### 1. Install Dependencies
#@markdown This cell will install all required packages and define all models and training functions.
!pip install pesq[speechmetrics] pystoi librosa

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import os
import numpy as np
import torch.nn.functional as F
import math
import traceback
import tarfile
import requests
import threading
from google.colab import files # For downloading

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---------------------------------------------------------------------
# CONFIGURATION OPTIONS (This will become a UI in Colab)
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 2. Training Configuration
#@markdown Select your model and dataset. The dataset will be downloaded automatically.
MODEL_ARCHITECTURE = "TS3_Codec (16kbps, Transformer)" #@param ["TS3_Codec (16kbps, Transformer)", "GRU_Codec (16kbps, Fast)", "ScoreDec Post-Filter (on GRU Codec)"]
DATASET_TO_DOWNLOAD = "train-clean-100" #@param ["dev-clean", "train-clean-100", "train-clean-360"]
#@markdown *Note: `dev-clean` is small and good for testing. Use `train-clean-100` for a real model.*

#@markdown ---
#@markdown ### 3. Training Hyperparameters
EPOCHS = 100 #@param {type:"integer"}
#@markdown (You can set this high and stop the script manually after 6 hours)
LEARNING_RATE = 0.0002 #@param {type:"number"}

#@markdown ---
#@markdown ### 4. Hardware & VRAM Settings
#@markdown A Colab T4 GPU has 16GB VRAM. You can use a larger batch size than on your 1060.
BATCH_SIZE = 16 #@param {type:"integer"}
#@markdown (Batch Size * Accum Steps = Effective Batch Size)
GRAD_ACCUM_STEPS = 2 #@param {type:"integer"}
USE_AMP = True #@param {type:"boolean"}
NUM_WORKERS = 4 #@param {type:"integer"}

#@markdown ---
#@markdown ### 5. File Paths
SAVE_PATH_PREFIX = "low_latency_codec" #@param {type:"string"}
DOWNLOAD_PATH = "/content/LibriSpeech" #@param {type:"string"}
#@markdown ---

# ---------------------------------------------------------------------
# DATASET DOWNLOAD FUNCTION
# ---------------------------------------------------------------------

def download_librispeech(dataset_name, path):
    """Downloads and extracts a LibriSpeech dataset."""
    url_map = {
        "dev-clean": "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "train-clean-100": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
        "train-clean-360": "https://www.openslr.org/resources/12/train-clean-360.tar.gz",
    }
    if dataset_name not in url_map:
        print(f"Error: Dataset '{dataset_name}' not recognized.")
        return None

    url = url_map[dataset_name]
    filename = url.split("/")[-1]
    compressed_path = os.path.join(path, filename)

    # FIX: Keep the original name with hyphens (LibriSpeech extracts with hyphens, not underscores)
    target_folder_name = dataset_name  # Use original name: 'train-clean-100'

    os.makedirs(path, exist_ok=True)

    # The extracted path will be /content/LibriSpeech/LibriSpeech/train-clean-100
    expected_dataset_path = os.path.join(path, "LibriSpeech", target_folder_name)

    print(f"Checking for existing dataset folder at: {expected_dataset_path}")
    if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
        print(f"Dataset already found at: {expected_dataset_path}")
        return expected_dataset_path

    try:
        # Download if needed
        if not os.path.exists(compressed_path):
            print(f"Dataset not found. Downloading {filename}...")
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                total_size = int(r.headers.get('content-length', 0))
                print(f"Total size: {total_size / (1024**3):.2f} GB")
                with open(compressed_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
            print("Download complete.")
        else:
             print(f"Compressed file {filename} already exists. Skipping download.")

        print("Extracting... (this may take several minutes)")
        with tarfile.open(compressed_path, "r:gz") as tar:
            tar.extractall(path=path)
        print("Extraction complete.")
        os.remove(compressed_path)

        # Verify the extracted folder exists
        print(f"Verifying extracted folder at: {expected_dataset_path}")
        if os.path.exists(expected_dataset_path) and os.path.isdir(expected_dataset_path):
            print(f"✓ Dataset ready at: {expected_dataset_path}")
            return expected_dataset_path
        else:
            print(f"ERROR: Could not find extracted folder at: {expected_dataset_path}")
            print(f"Contents of {path}:")
            os.system(f"ls -la {path}")
            if os.path.exists(os.path.join(path, "LibriSpeech")):
                 print(f"Contents of {os.path.join(path, 'LibriSpeech')}:")
                 os.system(f"ls -la {os.path.join(path, 'LibriSpeech')}")
            return None

    except Exception as e:
        print(f"Error during download/extraction: {e}")
        traceback.print_exc()
        return None        # --- End Specific Check ---


    except Exception as e:
        print(f"Error during download/extraction: {e}")
        traceback.print_exc()
        return None # Return None on error

# ---------------------------------------------------------------------
# MODEL DEFINITIONS (Copied *exactly* from your VRAM-Optimized model.py)
# ---------------------------------------------------------------------

class MultiResolutionSTFTLoss(nn.Module):
    def __init__(self, fft_sizes=[1024, 2048, 512], hop_sizes=[120, 240, 50], win_lengths=[600, 1200, 240]):
        super(MultiResolutionSTFTLoss, self).__init__()
        self.fft_sizes = fft_sizes
        self.hop_sizes = hop_sizes
        self.win_lengths = win_lengths
        self.window = torch.hann_window
        assert len(fft_sizes) == len(hop_sizes) == len(win_lengths)

    def forward(self, y_hat, y):
        sc_loss, mag_loss = 0.0, 0.0
        for fft, hop, win in zip(self.fft_sizes, self.hop_sizes, self.win_lengths):
            window = self.window(win, device=y.device)
            # Ensure input tensors are float32 for STFT
            y_hat_float = y_hat.squeeze(1).to(torch.float32)
            y_float = y.squeeze(1).to(torch.float32)
            spec_hat = torch.stft(y_hat_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)
            spec = torch.stft(y_float, n_fft=fft, hop_length=hop, win_length=win, window=window, return_complex=True)
            sc_loss += torch.norm(torch.abs(spec) - torch.abs(spec_hat), p='fro') / torch.norm(torch.abs(spec), p='fro')
            mag_loss += F.l1_loss(torch.log(torch.abs(spec).clamp(min=1e-9)), torch.log(torch.abs(spec_hat).clamp(min=1e-9)))
        return (sc_loss / len(self.fft_sizes)) + (mag_loss / len(self.fft_sizes))

class CausalConv1d(nn.Conv1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - 1
    def forward(self, x):
        return super().forward(F.pad(x, (self.causal_padding, 0)))

class CausalConvTranspose1d(nn.ConvTranspose1d):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.causal_padding = self.kernel_size[0] - self.stride[0]
    def forward(self, x):
        x = super().forward(x)
        if self.causal_padding != 0:
            return x[..., :-self.causal_padding]
        return x

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super(VectorQuantizer, self).__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.embedding = nn.Embedding(self.num_embeddings, self.embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / self.num_embeddings, 1.0 / self.num_embeddings)

    def forward(self, z_e):
        # Ensure input is float32 for distance calculation
        z_e_float = z_e.to(torch.float32)
        z_e_flat = z_e_float.permute(0, 2, 1).contiguous().view(-1, self.embedding_dim)
        distances = (torch.sum(z_e_flat**2, dim=1, keepdim=True)
                     + torch.sum(self.embedding.weight**2, dim=1)
                     - 2 * torch.matmul(z_e_flat, self.embedding.weight.t()))
        encoding_indices = torch.argmin(distances, dim=1)
        z_q = self.embedding(encoding_indices).view(z_e_float.shape[0], -1, self.embedding_dim)
        z_q = z_q.permute(0, 2, 1).contiguous()
        # Ensure losses are calculated with float32
        e_loss = F.mse_loss(z_q.detach(), z_e_float) * self.commitment_cost
        q_loss = F.mse_loss(z_q, z_e_float.detach())
        vq_loss = q_loss + e_loss
        # Apply STE using original input dtype (important for AMP)
        z_q = z_e + (z_q - z_e_float).detach()
        return z_q, vq_loss, encoding_indices.view(z_e.shape[0], -1)

# --- VRAM-Optimized Model Architecture ---
HOP_SIZE = 320
LATENT_DIM = 48 # Using the 48-dim version for 100% compatibility
VQ_EMBEDDINGS = 256
NUM_QUANTIZERS = 40

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(1, 32, 7), nn.ELU(),
            CausalConv1d(32, 64, 5, stride=2), nn.ELU(),
            CausalConv1d(64, 64, 5, stride=2), nn.ELU(),
            CausalConv1d(64, LATENT_DIM, 5, stride=2), nn.ELU()
        )
    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            CausalConvTranspose1d(LATENT_DIM, 64, 5, stride=2), nn.ELU(),
            CausalConvTranspose1d(64, 64, 5, stride=2), nn.ELU(),
            CausalConvTranspose1d(64, 32, 5, stride=2), nn.ELU(),
            CausalConv1d(32, 1, 7), nn.Tanh()
        )
    def forward(self, x):
        return self.net(x)

class CausalTransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, num_layers):
        super().__init__()
        self.d_model = d_model
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, activation=F.gelu) # Added GELU
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)

    def get_causal_mask(self, sz, device):
        return torch.triu(torch.ones(sz, sz, device=device), diagonal=1).to(torch.bool) # Ensure boolean mask

    def forward(self, x, state):
        inp = x if state is None else torch.cat([state, x], dim=1)
        STATE_LEN = 400
        if inp.shape[1] > STATE_LEN:
            inp = inp[:, -STATE_LEN:, :]
        new_state = inp.detach()
        mask = self.get_causal_mask(inp.shape[1], x.device)
        # Add layer norm before transformer
        norm_inp = F.layer_norm(inp, inp.shape[-1:])
        out = self.transformer(norm_inp, mask=mask, src_key_padding_mask=None) # Pass mask explicitly
        out = out[:, -x.shape[1]:, :]
        return out, new_state

class GRU_Codec(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.quantizer = VectorQuantizer(VQ_EMBEDDINGS, LATENT_DIM, 0.25)
        self.decoder = Decoder()
        self.encoder_rnn = nn.GRU(LATENT_DIM, LATENT_DIM, batch_first=True)
        self.decoder_rnn = nn.GRU(LATENT_DIM, LATENT_DIM, batch_first=True)

    def forward(self, x, h_enc=None, h_dec=None):
        z_e = self.encoder(x)
        z_e_rnn_in = z_e.permute(0, 2, 1)
        z_e_rnn_out, h_enc_new = self.encoder_rnn(z_e_rnn_in, h_enc)
        z_e_rnn_out = z_e_rnn_out.permute(0, 2, 1)
        z_q, vq_loss, indices = self.quantizer(z_e_rnn_out)
        z_q_rnn_in = z_q.permute(0, 2, 1)
        z_q_rnn_out, h_dec_new = self.decoder_rnn(z_q_rnn_in, h_dec)
        z_q_rnn_out = z_q_rnn_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_rnn_out)
        return x_hat, vq_loss, (h_enc_new, h_dec_new)

    def encode(self, x, h_enc):
        z_e = self.encoder(x)
        z_e_rnn_in = z_e.permute(0, 2, 1)
        z_e_rnn_out, h_enc_new = self.encoder_rnn(z_e_rnn_in, h_enc)
        z_e_rnn_out = z_e_rnn_out.permute(0, 2, 1)
        _, _, indices = self.quantizer(z_e_rnn_out)
        return indices, h_enc_new

    def decode(self, indices, h_dec):
        z_q = self.quantizer.embedding(indices).permute(0, 2, 1) # Directly permute after embedding
        z_q_rnn_in = z_q.permute(0, 2, 1)
        z_q_rnn_out, h_dec_new = self.decoder_rnn(z_q_rnn_in, h_dec)
        z_q_rnn_out = z_q_rnn_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_rnn_out)
        return x_hat, h_dec_new

class TS3_Codec(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.quantizer = VectorQuantizer(VQ_EMBEDDINGS, LATENT_DIM, 0.25)
        self.decoder = Decoder()
        # Using the nhead=2 version for 100% compatibility
        self.encoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=2, num_layers=2)
        self.decoder_tfm = CausalTransformerEncoder(LATENT_DIM, nhead=2, num_layers=2)

    def forward(self, x, h_enc=None, h_dec=None):
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        z_q, vq_loss, indices = self.quantizer(z_e_tfm_out)
        z_q_tfm_in = z_q.permute(0, 2, 1)
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q_tfm_in, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, vq_loss, (h_enc_new, h_dec_new)

    def encode(self, x, h_enc):
        z_e = self.encoder(x)
        z_e_tfm_in = z_e.permute(0, 2, 1)
        z_e_tfm_out, h_enc_new = self.encoder_tfm(z_e_tfm_in, h_enc)
        z_e_tfm_out = z_e_tfm_out.permute(0, 2, 1)
        _, _, indices = self.quantizer(z_e_tfm_out)
        return indices, h_enc_new

    def decode(self, indices, h_dec):
        z_q = self.quantizer.embedding(indices) # Shape: (B, T_latent, C)
        # Transformer expects (B, T, C) - no permute needed before
        z_q_tfm_out, h_dec_new = self.decoder_tfm(z_q, h_dec)
        z_q_tfm_out = z_q_tfm_out.permute(0, 2, 1) # Permute for Decoder: (B, C, T_latent)
        x_hat = self.decoder(z_q_tfm_out)
        return x_hat, h_dec_new

# --- ScoreDec Models (for compatibility) ---

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.dim = dim
    def forward(self, x):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=x.device) * -emb)
        emb = x[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class DiffusionUNet1D(nn.Module):
    def __init__(self, in_channels=1, model_channels=64, time_emb_dim=256, cond_dim=1):
        super().__init__()
        self.time_mlp_simple = nn.Sequential(SinusoidalPosEmb(time_emb_dim), nn.Linear(time_emb_dim, model_channels), nn.ReLU())
        self.cond_proj_simple = CausalConv1d(cond_dim, model_channels, 1)
        self.in_conv = CausalConv1d(in_channels, model_channels, 1)
        self.blocks = nn.ModuleList([CausalConv1d(model_channels, model_channels, 3) for _ in range(4)])
        self.out_conv = CausalConv1d(model_channels, in_channels, 1)
    def forward(self, x, time, cond):
        t_emb = self.time_mlp_simple(time).unsqueeze(-1)
        c_emb = self.cond_proj_simple(cond)
        x_in = self.in_conv(x)
        h = x_in + t_emb + c_emb
        for block in self.blocks:
            h = block(h) + h
        return self.out_conv(h)

class ScoreDecPostFilter(nn.Module):
    def __init__(self, timesteps=50, model_channels=64):
        super().__init__()
        self.timesteps = timesteps
        self.model = DiffusionUNet1D(model_channels=model_channels)
        betas = torch.linspace(1e-4, 0.02, timesteps)
        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, axis=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas_cumprod', alphas_cumprod)
        self.register_buffer('alphas', alphas)

    def q_sample(self, x_start, t, noise=None):
        if noise is None: noise = torch.randn_like(x_start)
        sqrt_alphas_cumprod_t = self.alphas_cumprod[t].sqrt().view(x_start.shape[0], 1, 1)
        sqrt_one_minus_alphas_cumprod_t = (1. - self.alphas_cumprod[t]).sqrt().view(x_start.shape[0], 1, 1)
        noised_tensor = sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise
        return noised_tensor, noise

    @torch.no_grad()
    def p_sample(self, x_t, t, cond):
        t_tensor = torch.full((x_t.shape[0],), t, device=x_t.device, dtype=torch.long)
        alpha_t = self.alphas[t].view(-1, 1, 1)
        alpha_cumprod_t = self.alphas_cumprod[t].view(-1, 1, 1)
        predicted_noise = self.model(x_t, t_tensor.float(), cond)
        x_prev = (x_t - ((1-alpha_t) / (1-alpha_cumprod_t).sqrt()) * predicted_noise) / alpha_t.sqrt()
        if t > 0:
            noise = torch.randn_like(x_t)
            alpha_cumprod_prev_t = self.alphas_cumprod[t-1]
            posterior_variance = (1-alpha_cumprod_prev_t) / (1-alpha_cumprod_t) * self.betas[t]
            x_prev += torch.sqrt(posterior_variance.view(-1, 1, 1)) * noise
        return x_prev

    @torch.no_grad()
    def enhance(self, x_low_quality, timesteps=10):
        t_start = timesteps - 1
        x_t, _ = self.q_sample(x_low_quality, torch.tensor([t_start], device=x_low_quality.device))
        for i in reversed(range(timesteps)):
            x_t = self.p_sample(x_t, i, cond=x_low_quality)
        return torch.tanh(x_t)

# ---------------------------------------------------------------------
# DATASET AND TRAINING LOGIC (Copied *exactly* from your model.py)
# ---------------------------------------------------------------------

TRAIN_CHUNK_SIZE = 16000 # 1 second

class AudioChunkDataset(Dataset):
    def __init__(self, directory, chunk_size=TRAIN_CHUNK_SIZE, sample_rate=16000):
        self.chunk_size, self.sample_rate = chunk_size, sample_rate
        print(f"Searching for audio files in: {directory}")
        self.file_paths = [os.path.join(r, f) for r, _, fs in os.walk(directory) for f in fs if f.lower().endswith(('.wav', '.flac'))]
        if not self.file_paths: raise ValueError(f"No audio files found in directory: {directory}")
        print(f"Found {len(self.file_paths)} audio files.")

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        try:
            waveform, sr = torchaudio.load(self.file_paths[idx])
            if sr != self.sample_rate: waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)
            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            # Ensure waveform is float32 before padding/slicing
            waveform = waveform.to(torch.float32)
            if waveform.shape[1] > self.chunk_size:
                start = np.random.randint(0, waveform.shape[1] - self.chunk_size)
                waveform = waveform[:, start:start + self.chunk_size]
            else:
                waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
             # Final check for length consistency
            if waveform.shape[1] != self.chunk_size:
                 print(f"Warning: File {self.file_paths[idx]} resulted in wrong length {waveform.shape[1]} after processing. Padding again.")
                 waveform = F.pad(waveform, (0, self.chunk_size - waveform.shape[1]))
            return waveform
        except Exception as e:
            print(f"Warning: Skipping file {self.file_paths[idx]}. Error: {e}")
            return torch.zeros((1, self.chunk_size), dtype=torch.float32) # Return float32

def train_model(dataset_path, epochs, learning_rate, batch_size, model_save_path, progress_callback, stop_event, model_type, use_amp=True, accum_steps=1, num_workers=4):
    """
    Main training function with AMP and Gradient Accumulation.
    """
    try:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        progress_callback.emit(f"Using device: {device}")

        dataset = AudioChunkDataset(directory=dataset_path)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, drop_last=True)
        progress_callback.emit(f"Dataset loaded with {len(dataset)} files. Dataloader starting...")

        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

        if model_type == 'gru':
            model = GRU_Codec().to(device)
            optimizer = optim.Adam(model.parameters(), lr=learning_rate)
            stft_criterion = MultiResolutionSTFTLoss().to(device)
            l1_criterion = nn.L1Loss().to(device)
            progress_callback.emit(f"Starting training for GRU_Codec model...")

        elif model_type == 'transformer':
            model = TS3_Codec().to(device)
            optimizer = optim.Adam(model.parameters(), lr=learning_rate)
            stft_criterion = MultiResolutionSTFTLoss().to(device)
            l1_criterion = nn.L1Loss().to(device)
            progress_callback.emit(f"Starting training for TS3_Codec (Optimized) model...")

        elif model_type == 'scoredec':
            progress_callback.emit("--- Starting ScoreDec Post-Filter Training ---")
            progress_callback.emit("Loading pre-trained GRU_Codec...")
            gru_path = f"{SAVE_PATH_PREFIX}_gru.pth"
            if not os.path.exists(gru_path):
                 progress_callback.emit(f"ERROR: '{gru_path}' not found. You must train the GRU_Codec first.")
                 return

            gru_codec = GRU_Codec().to(device)
            gru_codec.load_state_dict(torch.load(gru_path, map_location=device))
            gru_codec.eval()
            for param in gru_codec.parameters():
                param.requires_grad = False
            progress_callback.emit("GRU_Codec loaded and frozen.")

            model = ScoreDecPostFilter().to(device)
            optimizer = optim.Adam(model.parameters(), lr=learning_rate)
            l1_criterion = nn.L1Loss().to(device)
            progress_callback.emit("Starting training for ScoreDec model...")

        else:
            raise ValueError(f"Unknown model type for training: {model_type}")

        model.train() # Ensure model is in training mode

        for epoch in range(epochs):
            if stop_event.is_set():
                progress_callback.emit("Training stopped by user."); break

            optimizer.zero_grad()
            total_loss_epoch = 0.0 # Track epoch loss

            for i, data in enumerate(dataloader):
                # We catch KeyboardInterrupt here to allow saving when user stops cell
                try:
                    if stop_event.is_set(): break

                    inputs = data.to(device, non_blocking=True).to(torch.float32) # Ensure float32 input
                    # Check for empty batches (can happen if file loading failed for all items)
                    if inputs.numel() == 0:
                        progress_callback.emit(f"Warning: Skipping empty batch at index {i}")
                        continue

                    with torch.cuda.amp.autocast(enabled=use_amp):
                        if model_type in ['gru', 'transformer']:
                            h_enc, h_dec = None, None
                            x_hat, vq_loss, _ = model(inputs, h_enc, h_dec)
                            input_len, output_len = inputs.shape[-1], x_hat.shape[-1]
                            if output_len < input_len:
                                x_hat = F.pad(x_hat, (0, input_len - output_len))
                            elif output_len > input_len:
                                x_hat = x_hat[..., :input_len]

                            stft_loss = stft_criterion(x_hat, inputs)
                            l1_loss = l1_criterion(x_hat, inputs)
                            loss = stft_loss + 0.1 * l1_loss + vq_loss

                        elif model_type == 'scoredec':
                            with torch.no_grad():
                                x_hat_low_quality, _, _ = gru_codec(inputs)
                                input_len, output_len = inputs.shape[-1], x_hat_low_quality.shape[-1]
                                if output_len < input_len:
                                    x_hat_low_quality = F.pad(x_hat_low_quality, (0, input_len - output_len))
                                elif output_len > input_len:
                                    x_hat_low_quality = x_hat_low_quality[..., :input_len]
                                x_hat_low_quality = x_hat_low_quality.detach()

                            t = torch.randint(0, model.timesteps, (inputs.shape[0],), device=device).long()
                            x_t, noise = model.q_sample(x_start=inputs, t=t)
                            predicted_noise = model.model(x_t, t.float(), cond=x_hat_low_quality)
                            loss = l1_criterion(predicted_noise, noise)

                    # Ensure loss is a scalar float
                    if not isinstance(loss, torch.Tensor) or loss.ndim != 0:
                        progress_callback.emit(f"Warning: Invalid loss type or shape at batch {i}: {type(loss)}, shape {loss.shape if isinstance(loss, torch.Tensor) else 'N/A'}. Skipping batch.")
                        continue

                    loss = loss / accum_steps
                    scaler.scale(loss).backward()
                    total_loss_epoch += loss.item() * accum_steps # Accumulate *unnormalized* loss

                    if (i + 1) % accum_steps == 0:
                        # Gradient Clipping
                        # scaler.unscale_(optimizer) # Unscale for clipping
                        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                        scaler.step(optimizer)
                        scaler.update()
                        optimizer.zero_grad()

                        log_freq = 20 * accum_steps # Log approx every 20 effective batches
                        if (i + 1) % log_freq == 0:
                            current_lr = optimizer.param_groups[0]['lr']
                            if model_type in ['gru', 'transformer']:
                                progress_callback.emit(f"[Epoch {epoch + 1}, Batch {i + 1}/{len(dataloader)}] Loss: {loss.item() * accum_steps:.5f} (STFT: {stft_loss.item():.4f}, VQ: {vq_loss.item():.4f}) LR: {current_lr:.1e}")
                            elif model_type == 'scoredec':
                                progress_callback.emit(f"[Epoch {epoch + 1}, Batch {i + 1}/{len(dataloader)}] Denoising Loss: {loss.item() * accum_steps:.5f} LR: {current_lr:.1e}")

                except KeyboardInterrupt:
                    progress_callback.emit("\n!!! KeyboardInterrupt received. Stopping training... !!!")
                    stop_event.set()
                    break
                except Exception as batch_error:
                    progress_callback.emit(f"!!! ERROR in batch {i+1}. Skipping batch. Error: {batch_error} !!!")
                    # Optionally add traceback print here for more detail
                    # traceback.print_exc()
                    optimizer.zero_grad() # Clear any potentially corrupted gradients


            avg_loss_epoch = total_loss_epoch / len(dataloader)
            progress_callback.emit(f"--- Epoch {epoch + 1} finished --- Average Loss: {avg_loss_epoch:.5f} ---")


            # Handle potential leftover gradients if dataloader size is not multiple of accum_steps
            is_final_step = (i + 1) % accum_steps != 0
            if is_final_step and not stop_event.is_set():
                progress_callback.emit("Performing final optimizer step for epoch...")
                try:
                    # scaler.unscale_(optimizer) # Unscale for clipping
                    # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                except Exception as final_step_e:
                     progress_callback.emit(f"Error during final step: {final_step_e}")


            # --- Save model at the end of *every* epoch ---
            if not stop_event.is_set():
                progress_callback.emit(f"Saving model checkpoint to {model_save_path}...")
                try:
                    torch.save(model.state_dict(), model_save_path)
                    progress_callback.emit(f"Checkpoint saved.")
                except Exception as save_e:
                    progress_callback.emit(f"!!! Error saving checkpoint: {save_e} !!!")

        if not stop_event.is_set():
            progress_callback.emit("Training finished.")
        else:
            progress_callback.emit("Training stopped.")

        progress_callback.emit(f"Final model is saved at {model_save_path}")

    except Exception as e:
        progress_callback.emit(f"ERROR in training setup or outer loop: {e}")
        progress_callback.emit(traceback.format_exc())

    # --- Final save attempt in case of stop/interrupt ---
    finally:
        if 'model' in locals() and model is not None:
             try:
                 progress_callback.emit(f"Attempting final save to {model_save_path}...")
                 torch.save(model.state_dict(), model_save_path)
                 progress_callback.emit(f"Final model state saved successfully.")
             except Exception as final_save_e:
                 progress_callback.emit(f"!!! Error during final save: {final_save_e} !!!")
        elif 'model' in locals() and model is None:
             progress_callback.emit("Model object exists but is None. Cannot save.")
        else:
            progress_callback.emit(f"Model variable not found in local scope. Cannot save.")


# ---------------------------------------------------------------------
# MAIN EXECUTION SCRIPT
# ---------------------------------------------------------------------
#@markdown ---
#@markdown ### 6. Run Training
#@markdown Click the "Run" button below (or Shift+Enter in this cell) to start the training process using the settings above.
print("Starting main training script...")

# 1. Map GUI string to internal model_type
model_type_map = {
    "GRU_Codec (16kbps, Fast)": "gru",
    "TS3_Codec (16kbps, Transformer)": "transformer",
    "ScoreDec Post-Filter (on GRU Codec)": "scoredec"
}
model_type_internal = model_type_map[MODEL_ARCHITECTURE]

# 2. Update save path
final_save_path = f"{SAVE_PATH_PREFIX}_{model_type_internal}.pth"
print(f"Selected model: {MODEL_ARCHITECTURE}")
print(f"Model will be saved to: {final_save_path}")

# 3. Download data
print(f"Downloading/Verifying {DATASET_TO_DOWNLOAD} in {DOWNLOAD_PATH}...")
# --- Use the returned path from download function ---
actual_dataset_path = download_librispeech(DATASET_TO_DOWNLOAD, DOWNLOAD_PATH)

if actual_dataset_path: # Check if download/extraction was successful
    print(f"Dataset ready at: {actual_dataset_path}")

    # 4. Check for ScoreDec dependency
    if model_type_internal == 'scoredec':
        gru_path = f"{SAVE_PATH_PREFIX}_gru.pth"
        print(f"Checking for ScoreDec dependency: '{gru_path}'...")
        if not os.path.exists(gru_path):
            print(f"--- ERROR ---")
            print(f"You must train the 'GRU_Codec' first and save '{gru_path}' before training ScoreDec.")
            print(f"Please change MODEL_ARCHITECTURE to 'GRU_Codec' and run again first.")
            # Stop execution
            raise SystemExit("Dependency not found.")
        print("Dependency found.")

    # 5. Start training
    print("--- Starting Training ---")
    # Using a dummy stop_event for Colab, as it's not interactive
    stop_event = threading.Event()

    # A simple print function for the callback
    class ColabProgressEmitter:
        def emit(self, msg):
            print(msg)

    colab_printer = ColabProgressEmitter()

    train_model(
        dataset_path=actual_dataset_path, # Use the found path
        epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        model_save_path=final_save_path,
        progress_callback=colab_printer,
        stop_event=stop_event,
        model_type=model_type_internal,
        use_amp=USE_AMP,
        accum_steps=GRAD_ACCUM_STEPS,
        num_workers=NUM_WORKERS
    )

    print("--- Training Finished ---")
    print(f"Model saved to {final_save_path}")
    print("\n=======================================================")
    print("To download your model, run the next cell.")
    print("=======================================================")

else:
    print("Failed to download/find dataset. Training aborted.")



PyTorch Version: 2.8.0+cu126
CUDA Available: True
GPU: Tesla T4
Starting main training script...
Selected model: TS3_Codec (16kbps, Transformer)
Model will be saved to: low_latency_codec_transformer.pth
Downloading/Verifying train-clean-100 in /content/LibriSpeech...
Checking for existing dataset folder at: /content/LibriSpeech/LibriSpeech/train-clean-100
Dataset already found at: /content/LibriSpeech/LibriSpeech/train-clean-100
Dataset ready at: /content/LibriSpeech/LibriSpeech/train-clean-100
--- Starting Training ---
Using device: cuda
Searching for audio files in: /content/LibriSpeech/LibriSpeech/train-clean-100
Found 28539 audio files.
Dataset loaded with 28539 files. Dataloader starting...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/tmp/ipython-input-2750495032.py:456: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Starting training for TS3_Codec (Optimized) model...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/pytho

[Epoch 1, Batch 40/1783] Loss: 15.69981 (STFT: 14.4492, VQ: 1.2472) LR: 2.0e-04
[Epoch 1, Batch 80/1783] Loss: 15.77856 (STFT: 14.5337, VQ: 1.2410) LR: 2.0e-04
[Epoch 1, Batch 120/1783] Loss: 15.54253 (STFT: 14.3045, VQ: 1.2345) LR: 2.0e-04
[Epoch 1, Batch 160/1783] Loss: 15.87712 (STFT: 14.6463, VQ: 1.2274) LR: 2.0e-04
[Epoch 1, Batch 200/1783] Loss: 15.96895 (STFT: 14.7441, VQ: 1.2208) LR: 2.0e-04
[Epoch 1, Batch 240/1783] Loss: 15.77192 (STFT: 14.5557, VQ: 1.2127) LR: 2.0e-04
[Epoch 1, Batch 280/1783] Loss: 15.65180 (STFT: 14.4430, VQ: 1.2054) LR: 2.0e-04
[Epoch 1, Batch 320/1783] Loss: 15.66648 (STFT: 14.4658, VQ: 1.1979) LR: 2.0e-04
[Epoch 1, Batch 360/1783] Loss: 15.61312 (STFT: 14.4198, VQ: 1.1901) LR: 2.0e-04
[Epoch 1, Batch 400/1783] Loss: 15.92139 (STFT: 14.7348, VQ: 1.1828) LR: 2.0e-04
[Epoch 1, Batch 440/1783] Loss: 15.19137 (STFT: 14.0138, VQ: 1.1751) LR: 2.0e-04
[Epoch 1, Batch 480/1783] Loss: 15.73438 (STFT: 14.5621, VQ: 1.1678) LR: 2.0e-04
[Epoch 1, Batch 520/1783] Loss

In [ ]:
# ---------------------------------------------------------------------
# CELL 3: RUN THIS CELL *AFTER* TRAINING IS FINISHED (OR STOPPED)
#         TO DOWNLOAD YOUR .pth MODEL FILE
# ---------------------------------------------------------------------

#@title Download Your Trained Model

import os
from google.colab import files

# --- Determine the model file to download based on Cell 2's settings ---
# (Duplicating logic slightly to ensure correct filename)
model_type_map_dl = {
    "GRU_Codec (16kbps, Fast)": "gru",
    "TS3_Codec (16kbps, Transformer)": "transformer",
    "ScoreDec Post-Filter (on GRU Codec)": "scoredec"
}
# Use the values from Cell 2's form (these need to be defined in Cell 2 scope)
# If running this cell independently, manually set these variables:
# MODEL_ARCHITECTURE = "TS3_Codec (16kbps, Transformer)" # Or GRU_Codec...
# SAVE_PATH_PREFIX = "low_latency_codec"

try:
    model_type_internal_dl = model_type_map_dl[MODEL_ARCHITECTURE]
    model_filename = f"{SAVE_PATH_PREFIX}_{model_type_internal_dl}.pth"

    print(f"Attempting to download: {model_filename}")

    if os.path.exists(model_filename):
        print("Model file found. Starting download...")
        files.download(model_filename)
        print("Download initiated.")
    else:
        print(f"ERROR: Model file '{model_filename}' not found!")
        print("Did the training in Cell 2 complete successfully or save a checkpoint?")
        print("Check the file browser on the left to see available .pth files.")

except NameError:
    print("ERROR: Could not determine the model filename.")
    print("Please ensure you have run Cell 2 successfully before running this cell,")
    print("OR manually define MODEL_ARCHITECTURE and SAVE_PATH_PREFIX in this cell.")
except Exception as e:
    print(f"An error occurred during download preparation: {e}")

